# Práctica: Implementación y Análisis de un Sistema de Recomendación Basado en Modelos (SVD)

**Asignatura:** Gestión de la Información  
**Grupo de prácticas:** Grupo 12 (Adrián Bello Sánchez y Alberto Cuervo Prieto)  
**Dataset:** MovieLens 100k (`ml-latest-small`)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from surprise import Reader, Dataset, SVD
from surprise.model_selection import GridSearchCV
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

## 1. Implementación básica del filtro colaborativo (SVD con Grid Search y CV)

In [2]:
ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

param_grid = {
    'n_factors': [20, 50, 100],
    'n_epochs': [20, 30],
    'lr_all': [0.005, 0.01],
    'reg_all': [0.05, 0.1]
}

gs = GridSearchCV(SVD, param_grid, measures=['RMSE', 'MAE'], cv=3, n_jobs=-1)
gs.fit(data)

print(f"Mejor RMSE: {gs.best_score['rmse']:.4f}")
print(f"Mejor MAE:  {gs.best_score['mae']:.4f}")
print('\nMejores hiperparámetros:')
for param, valor in gs.best_params['rmse'].items():
    print(f"  - {param}: {valor}")

params = gs.best_params['rmse']
modelo = SVD(n_factors=params['n_factors'], n_epochs=params['n_epochs'],
             lr_all=params['lr_all'], reg_all=params['reg_all'], random_state=42)

Mejor RMSE: 0.8628
Mejor MAE:  0.6625

Mejores hiperparámetros:
  - n_factors: 100
  - n_epochs: 30
  - lr_all: 0.01
  - reg_all: 0.1


## 2. Añadir usuarios del equipo y Top 10 recomendaciones

In [3]:
adrian_ratings = [
    {'userId': 9991, 'movieId': 2571, 'rating': 5.0},   # Matrix
    {'userId': 9991, 'movieId': 79132, 'rating': 5.0},  # Inception
    {'userId': 9991, 'movieId': 109487, 'rating': 5.0}, # Interstellar
    {'userId': 9991, 'movieId': 296, 'rating': 4.5},    # Pulp Fiction
    {'userId': 9991, 'movieId': 589, 'rating': 4.5},    # Terminator 2
    {'userId': 9991, 'movieId': 260, 'rating': 5.0},    # Star Wars IV
    {'userId': 9991, 'movieId': 1196, 'rating': 5.0},   # Star Wars V
    {'userId': 9991, 'movieId': 367, 'rating': 1.5},    # The Mask
    {'userId': 9991, 'movieId': 357, 'rating': 1.5},    # Four Weddings
    {'userId': 9991, 'movieId': 597, 'rating': 1.0},    # Pretty Woman
]

alberto_ratings = [
    {'userId': 9992, 'movieId': 1, 'rating': 5.0},      # Toy Story
    {'userId': 9992, 'movieId': 3114, 'rating': 5.0},   # Toy Story 2
    {'userId': 9992, 'movieId': 78499, 'rating': 5.0},  # Toy Story 3
    {'userId': 9992, 'movieId': 364, 'rating': 5.0},    # The Lion King
    {'userId': 9992, 'movieId': 595, 'rating': 4.5},    # Beauty and the Beast
    {'userId': 9992, 'movieId': 588, 'rating': 4.5},    # Aladdin
    {'userId': 9992, 'movieId': 6377, 'rating': 5.0},   # Finding Nemo
    {'userId': 9992, 'movieId': 589, 'rating': 1.0},    # Terminator 2
    {'userId': 9992, 'movieId': 1258, 'rating': 1.5},   # The Shining
]

df_equipo = pd.DataFrame(adrian_ratings + alberto_ratings)
ratings_ampliado = pd.concat([ratings[['userId', 'movieId', 'rating']], df_equipo], ignore_index=True)

data_ampliada = Dataset.load_from_df(ratings_ampliado, reader)
trainset = data_ampliada.build_full_trainset()
modelo.fit(trainset)

def recomendar_top_10(user_id, min_votos=20):
    vistas = set(ratings_ampliado[ratings_ampliado['userId'] == user_id]['movieId'])
    conteo_votos = ratings_ampliado['movieId'].value_counts()
    candidatas = conteo_votos[conteo_votos >= min_votos].index
    
    predicciones = []
    for mid in candidatas:
        if mid not in vistas:
            pred = modelo.predict(user_id, mid).est
            predicciones.append((mid, pred))
    
    predicciones.sort(key=lambda x: x[1], reverse=True)
    
    resultado = []
    for rank, (mid, nota) in enumerate(predicciones[:10], 1):
        info = movies[movies['movieId'] == mid].iloc[0]
        resultado.append({
            'Ranking': f"#{rank}",
            'Título': info['title'],
            'Géneros': info['genres'],
            'Predicción': round(nota, 2)
        })
    return pd.DataFrame(resultado)

print("=== TOP 10 RECOMENDACIONES: ADRIÁN (userId 9991) ===")
print(recomendar_top_10(9991).to_string(index=False))

print("\n=== TOP 10 RECOMENDACIONES: ALBERTO (userId 9992) ===")
print(recomendar_top_10(9992).to_string(index=False))

=== TOP 10 RECOMENDACIONES: ADRIÁN (userId 9991) ===
Ranking                                                                      Título                Géneros  Predicción
     #1                                                       Godfather, The (1972)            Crime|Drama        4.56
     #2 Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)             Comedy|War        4.44
     #3                                                  Usual Suspects, The (1995) Crime|Mystery|Thriller        4.39
     #4                                            Shawshank Redemption, The (1994)            Crime|Drama        4.37
     #5                   Lord of the Rings: The Fellowship of the Ring, The (2001)      Adventure|Fantasy        4.36
     #6                                                    On the Waterfront (1954)            Crime|Drama        4.35
     #7                                        Bridge on the River Kwai, The (1957)    Adventure|Drama|War        